# Day 10 — Startup Evolution Planner & Scenario Simulator (Layers 9 & 10)

Two capstone layers, both built on the **same warm Day-8 stack** (one model in memory):

- **Layer 9 — Evolution Planner:** classify a campaign's funding-**readiness** stage, then give a
  pre-launch **preparation** plan three ways — an ordered roadmap, ranked milestones, and a
  countdown timeline — each annotated with the readiness stage it unlocks.
- **Layer 10 — Scenario Simulator:** generate multi-variable **what-if scenarios**, score each
  through the real model, and **compare** them — rank, marginal value, best vs most-efficient.

**Honesty.** Kickstarter is a single **pre-launch snapshot**, so there is no MVP→PMF lifecycle
and no 12-month growth journey to observe. "Stage" means **funding readiness** (Not-ready /
Fixable / Ready / Strong from the calibrated probability + the reachable optimum); "roadmap /
timeline" mean **pre-launch preparation** over real controllable levers (goal / duration / launch
timing / title). Founder, retention, revenue, market-size and named-competitor items are
**deferred** — the data can't support them. Model-simulated, **not causal**. Logic in
`src/evolution/` and `src/scenario/`.

In [1]:
import sys, os, json, warnings, logging
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir(os.path.abspath(".."))
sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore"); logging.disable(logging.WARNING)
import pandas as pd, numpy as np

# ONE warm stack (Day-4 model + Day-7 frame + Day-8 counterfactual) feeds BOTH Day-10 layers. No faiss.
from src.counterfactual.pipeline import CounterfactualPipeline
from src.evolution.pipeline import EvolutionPipeline
from src.scenario.pipeline import ScenarioPipeline
cf  = CounterfactualPipeline()
evo = EvolutionPipeline(cf_pipeline=cf)
scn = ScenarioPipeline(cf_pipeline=cf)
assert evo.cf is cf is scn.simulator.cf            # both layers, one model in memory

# An improvable demo campaign (below the bar as-is, fixable before launch).
df = pd.read_csv("data/processed/kickstarter_processed.csv"); ti = np.load("artifacts/splits/test_index.npy")
row = df.iloc[ti][df.iloc[ti]["usd_goal_real"] > 12000].sample(1, random_state=4).iloc[0].to_dict()
E = evo.run(row)                                   # stage + roadmap + milestones + timeline (ONE analyze)
S = scn.run(row)                                   # generate + simulate + compare

print("CAMPAIGN: %s | %s | goal $%.0f" % (row["name"], row["main_category"], row["usd_goal_real"]))
print("readiness: %s  %.0f%% as-is -> %.0f%% optimized  (funding bar %.0f%%)" %
      (E["stage"]["stage"], E["stage"]["funding_probability"]*100,
       E["stage"]["optimized_probability"]*100, E["stage"]["threshold"]*100))

CAMPAIGN: Artificer - Board Game | Games | goal $30516
readiness: Fixable  24% as-is -> 66% optimized  (funding bar 33%)


## Phase B — Funding-Readiness Stage

Not a startup lifecycle stage (unobservable here) — a **funding-readiness** stage derived from
three real numbers: the calibrated probability, the decision threshold, and the Day-8 reachable
optimum. **Fixable** = below the bar as-is, but controllable changes can clear it before launch.

In [2]:
s = E["stage"]
print("STAGE:", s["stage"], "(confidence %.2f)" % s["confidence"])
print(s["reason"])

STAGE: Fixable (confidence 0.90)
As-is the model estimates 24% (below the 33% bar), but controllable changes could reach ~66% — above the bar. Fixable before launch.


## Phase C — Pre-Launch Preparation Roadmap

The Day-8 optimization plan, ordered current → readiness, each step annotated with the stage
**after** it — so a founder sees exactly *when* the campaign crosses the funding bar.

In [3]:
r = E["roadmap"]
print("%s (%.0f%%)  ->  %s (%.0f%%)\n" % (r["current_stage"], r["current_probability"]*100,
                                          r["target_stage"], r["target_probability"]*100))
for st in r["steps"]:
    bar = "   <- crosses funding bar" if st["crosses_bar"] else ""
    print("%d. %-46s %5.0f%%  %-8s%s" % (st["order"], st["action"],
          st["cumulative_probability"]*100, st["stage_after"], bar))
print("\n" + r["summary"])

Fixable (24%)  ->  Strong (66%)

1. Shorten your campaign from 29 to 15 days          54%  Strong     <- crosses funding bar
2. Lower your funding goal from $30,516 to $5,000    62%  Strong     <- crosses funding bar
3. Add a number to your title (e.g. an edition or volume number)    66%  Strong     <- crosses funding bar

Currently Fixable (24%). A 3-step preparation roadmap — starting with 'shorten your campaign from 29 to 15 days' — reaches Strong (~66%). You cross the funding bar at step 1.


## Phase D — Ranked Milestones

The same levers as milestones, each with the four ranking dimensions: predicted impact, execution
difficulty, historical evidence, funding influence. One per feature (best-ranked), no re-ranking —
so a milestone and its Day-8 recommendation agree. Product / team / traction milestones deferred.

In [4]:
for m in E["milestones"]["milestones"]:
    ev = (m["historical_evidence"] or "")[:58]
    print("P%d  %-34s  %+dpts  [diff %-6s]  %s" %
          (m["priority"], m["milestone"], m["predicted_impact_points"], m["execution_difficulty"], ev))

P1  Choose an optimal campaign length   +30pts  [diff Low   ]  Games campaigns with a <=15d campaign funded at 66% vs a 4
P2  Set a fundable funding goal         +12pts  [diff Low   ]  Games campaigns asking goal <= $5,000 funded at 55% vs a 4
P3  Strengthen your campaign title      +4pts  [diff Medium]  Games campaigns with a number in the title funded at 59% v
P4  Time your launch well               +3pts  [diff Low   ]  Games campaigns launched on Tuesday funded at 50% vs a 40%


## Phase H — Funding-Readiness Timeline

The **same** controllable levers as the roadmap and milestones — a third framing, placed on a
pre-launch **countdown** (Week -4 → Launch day). Not a multi-month post-launch growth timeline
(that needs longitudinal data we don't have).

In [5]:
t = E["timeline"]
for c in t["checkpoints"]:
    bar = "   <- crosses bar" if c["crosses_bar"] else ""
    print("%-9s  %-46s %5.0f%%  %-8s%s" % (c["when"], c["task"],
          c["cumulative_probability"]*100, c["readiness_after"], bar))
print("\nSame lever set as roadmap/milestones, on a time axis. Lead window: %d weeks." % t["lead_weeks"])

Week -4    Shorten your campaign from 29 to 15 days          54%  Strong     <- crosses bar
Week -3    Lower your funding goal from $30,516 to $5,000    62%  Strong     <- crosses bar
Week -1    Add a number to your title (e.g. an edition or volume number)    66%  Strong     <- crosses bar
Launch day  Launch the campaign                               66%  Strong     <- crosses bar

Same lever set as roadmap/milestones, on a time axis. Lead window: 4 weeks.


## Phases E & F — Scenarios, Simulated

The genuinely new capability over Day-8's single optimum: **multi-variable what-if scenarios**,
composed canonically (dependency-ordered, not a naive raw-edit merge), each scored through the
**real** model with a fresh call — never a sum of gains.

In [6]:
print("current %.0f%%  |  funding bar %.0f%%\n" % (S["current_probability"]*100, S["threshold"]*100))
for sc in S["ranked_scenarios"]:
    bar = " READY" if sc["crosses_bar"] else ""
    print("#%d  %-50s %5.0f%% (%+.0f)  %-8s%s" % (sc["rank"], sc["name"],
          sc["probability"]*100, sc["gain"]*100, sc["stage"], bar))

current 24%  |  funding bar 33%

#1  Lower goal + shorten duration + Tuesday launch + add title number    69% (+44)  Strong   READY
#2  Lower goal + shorten duration + Tuesday launch        64% (+40)  Strong   READY
#3  Lower goal + shorten duration                         62% (+37)  Strong   READY
#4  Lower goal                                            36% (+12)  Ready    READY
#5  Current (as-is)                                       24% (+0)  Fixable 


## Phase G — Scenario Comparison

**Marginal value** = the gain of adding *one* change on top of the ones before it (the step
difference, not the cumulative gain — attributing the cumulative number to the last lever
multi-counts). Two recommendations a founder actually weighs: highest-probability vs fewest-changes
to Ready (the latter is exact — scenarios add one change at a time, so the first bar-crosser is the
smallest one that clears it).

In [7]:
print("MARGINAL VALUE of each added change (cumulative order):")
for mv in S["marginal_values"]:
    print("  %-18s %+5.0f  %s" % (mv["change_label"], mv["marginal_gain"]*100,
          "worth it" if mv["worth_it"] else "skip"))
print("\nbest      : %s (%.0f%%, %s)" % (S["best_scenario"]["name"],
      S["best_scenario"]["probability"]*100, S["best_scenario"]["stage"]))
print("efficient :", S["most_efficient_scenario"])
print("\n" + S["recommendation"])

MARGINAL VALUE of each added change (cumulative order):
  Lower goal           +12  worth it
  shorten duration     +26  worth it
  Tuesday launch        +3  worth it
  add title number      +4  worth it

best      : Lower goal + shorten duration + Tuesday launch + add title number (69%, Strong)
efficient : {'name': 'Lower goal', 'probability': 0.3587, 'n_changes': 1}

Fastest path to Ready: 'Lower goal' — 1 change(s), ~36%. Highest probability overall: 'Lower goal + shorten duration + Tuesday launch + add title number' (~69%).


## Coherence & the honesty boundary

Because every evolution module reads the **same** shared analysis, the current stage is
*structurally* identical across the stage detector, the roadmap, and the timeline — a property of
the object graph, not a test. And the planner records exactly what it **refuses to fabricate**.

In [8]:
labels = {E["stage"]["stage"], E["roadmap"]["current_stage"], E["timeline"]["current_stage"]}
print("current-stage consistent across stage / roadmap / timeline:", len(labels) == 1, labels)
print("\nDEFERRED — refused, no data to support (%d items):" % len(E["deferred"]))
for k in E["deferred"]:
    print("   -", k)

current-stage consistent across stage / roadmap / timeline: True {'Fixable'}

DEFERRED — refused, no data to support (9 items):
   - startup_lifecycle
   - growth_milestones
   - validate_customer_problem
   - improve_retention
   - grow_revenue
   - expand_founding_team
   - build_partnerships
   - improve_unit_economics
   - growth_timeline


## Phase J — Serialization (content-hash identity + verify)

The planner has **no trained weights** — it's a *configuration* over the Day-4 model and Day-8
engine. So we persist it as **JSON, not a pickle**: the content-hash of the config + readiness rule
is its identity, pinned to the SHA-256 of its dependency models and output artifacts. `verify()`
re-hashes everything and localizes any drift.

In [9]:
from src.evolution.serialization import EvolutionSerializer
ser = EvolutionSerializer().build(); ser.save()
v = ser.verify()
print(ser.describe())
print("verify all_ok:", v["all_ok"], "(%d/%d checks pass)" % (sum(v["checks"].values()), len(v["checks"])))

Evolution planner v1.0.0 (content ee3b51065319…): layers [9, 10]; 2 deps hashed, 9 artifacts, 9 items deferred.
verify all_ok: True (12/12 checks pass)


## Day 10 complete — Layers 9 & 10 ✅

**Evolution Planner** (readiness stage → roadmap → milestones → timeline) and **Scenario
Simulator** (generate → simulate → compare), both on one warm stack, one model-heavy call, with a
content-hash-verified manifest. Every output is a reframing of a **real** controllable signal;
lifecycle, growth, founder, retention, revenue and competitor claims are **deferred, not
invented**. Model-simulated, not causal.